# Analisis Data Penduduk Desa Rancajaya

## tl;dr

Sheet **ASLI** berisi **15.772** rekaman penduduk dengan NIK unik dan tanpa baris duplikat. Setelah delapan tanggal lahir dalam bentuk nomor serial Excel dinormalisasi, seluruh umur konsisten pada snapshot **11 Oktober 2023**. Notebook ini hanya menghasilkan agregat publik; nama, NIK, nomor KK, dan alamat tidak pernah ditampilkan.

## Context & Methods

### Key Assumptions

- Satu baris pada sheet `ASLI` mewakili satu penduduk.
- Baris tanpa nama atau NIK bukan rekaman penduduk dan dikeluarkan.
- Tanggal lahir berbentuk nomor serial Excel diubah menggunakan asal tanggal Excel `1899-12-30`.
- Kategori pendidikan dan pekerjaan dinormalisasi untuk menggabungkan variasi ejaan/spasi.

Sumber: `../Data/Data Penduduk Desa Rancajaya.xlsx`, sheet `ASLI`.

## 1. Load and normalize

In [1]:
from pathlib import Path
import pandas as pd

SOURCE = Path("../../Data/Data Penduduk Desa Rancajaya.xlsx")
SHEET = "ASLI"

def parse_excel_date(value):
    if isinstance(value, (int, float)) and pd.notna(value) and 1 <= value < 60000:
        return pd.Timestamp("1899-12-30") + pd.to_timedelta(value, unit="D")
    return pd.to_datetime(value, errors="coerce")

def normalize_text(series):
    return series.astype("string").str.upper().str.replace(r"\s+", " ", regex=True).str.strip()

raw = pd.read_excel(SOURCE, sheet_name=SHEET, dtype={"NIK": "string", "No. KK": "string"})
raw.columns = [str(column).strip() for column in raw.columns]
raw["NIK"] = raw["NIK"].str.replace(r"\.0$", "", regex=True).str.strip()
raw["No. KK"] = raw["No. KK"].str.replace(r"\.0$", "", regex=True).str.strip()

residents = raw[raw["Nama Lengkap"].notna() & raw["NIK"].notna()].copy()
residents["tanggal_lahir"] = residents["Tgl. Lahir"].map(parse_excel_date)
residents["umur"] = pd.to_numeric(residents["Umur (L)"], errors="coerce")
residents["jenis_kelamin"] = normalize_text(residents["L"])
residents["pendidikan"] = normalize_text(residents["Pendidikan (L)"])
residents["pekerjaan"] = normalize_text(residents["Pekerjaan (L)"])

residents.shape

(15772, 25)

## 2. Data-quality checks

In [2]:
quality_summary = pd.DataFrame([
    ("Baris sumber", len(raw)),
    ("Rekaman penduduk valid", len(residents)),
    ("Baris non-data dikeluarkan", len(raw) - len(residents)),
    ("NIK unik", residents["NIK"].nunique()),
    ("NIK duplikat", residents["NIK"].duplicated(keep=False).sum()),
    ("Nomor KK unik", residents["No. KK"].nunique()),
    ("Baris duplikat identik", residents.duplicated().sum()),
    ("Tanggal lahir tidak valid", residents["tanggal_lahir"].isna().sum()),
    ("Umur tidak valid", residents["umur"].isna().sum()),
], columns=["Pemeriksaan", "Nilai"])
quality_summary

,Pemeriksaan,Nilai
0,Baris sumber,15801
1,Rekaman penduduk valid,15772
2,Baris non-data dikeluarkan,29
3,NIK unik,15772
4,NIK duplikat,0
5,Nomor KK unik,5379
6,Baris duplikat identik,0
7,Tanggal lahir tidak valid,0
8,Umur tidak valid,0


## 3. Validate snapshot date

In [3]:
age_check = residents.dropna(subset=["tanggal_lahir", "umur"]).copy()
age_check["awal_rentang"] = age_check.apply(lambda row: row["tanggal_lahir"] + pd.DateOffset(years=int(row["umur"])), axis=1)
age_check["akhir_rentang"] = age_check.apply(lambda row: row["tanggal_lahir"] + pd.DateOffset(years=int(row["umur"]) + 1), axis=1)

snapshot_window = {
    "awal": age_check["awal_rentang"].max().date().isoformat(),
    "akhir_eksklusif": age_check["akhir_rentang"].min().date().isoformat(),
}
snapshot_window

{'awal': '2023-10-11', 'akhir_eksklusif': '2023-10-12'}

## 4. Aggregate public statistics

In [4]:
education = residents["pendidikan"].replace({
    "TIDAK/BLMSEKOLAH": "TIDAK/BLM SEKOLAH",
    "TAMATSD/SEDERAJAT": "TAMAT SD/SEDERAJAT",
    "BELUM TAMATSD/SEDERAJAT": "BELUM TAMAT SD/SEDERAJAT",
    "AKADEMI/DIPLO MA III/SARJANA MUDA": "AKADEMI/DIPLOMA III/SARJANA MUDA",
    "AKADEMI/DIPLO MA III/SARJANA": "AKADEMI/DIPLOMA III/SARJANA",
})
job = residents["pekerjaan"].replace({"PELAJAR/MAHASI SWA": "PELAJAR/MAHASISWA"})

age_groups = pd.cut(
    residents["umur"],
    bins=[-1, 5, 12, 17, 24, 44, 59, 200],
    labels=["0-5 tahun", "6-12 tahun", "13-17 tahun", "18-24 tahun", "25-44 tahun", "45-59 tahun", "60 tahun ke atas"],
).value_counts(sort=False)

public_summary = {
    "total_penduduk": len(residents),
    "laki_laki": int((residents["jenis_kelamin"] == "LAKI-LAKI").sum()),
    "perempuan": int((residents["jenis_kelamin"] == "PEREMPUAN").sum()),
    "kepala_keluarga": residents["No. KK"].nunique(),
    "rt": residents["RT"].nunique(),
    "rw": residents["RW"].nunique(),
    "kelompok_umur": age_groups.astype(int).to_dict(),
    "pendidikan": {
        "Tidak/Belum Sekolah": int((education == "TIDAK/BLM SEKOLAH").sum()),
        "Belum Tamat SD": int((education == "BELUM TAMAT SD/SEDERAJAT").sum()),
        "SD/Sederajat": int((education == "TAMAT SD/SEDERAJAT").sum()),
        "SMP/Sederajat": int((education == "SLTP/SEDERAJAT").sum()),
        "SMA/Sederajat": int((education == "SLTA/SEDERAJAT").sum()),
        "Diploma/Sarjana": int(education.isin(["DIPLOMA IV/STRATA I", "AKADEMI/DIPLOMA III/SARJANA MUDA", "DIPLOMA I/II", "STRATA-II", "AKADEMI/DIPLOMA III/SARJANA", "DIPLOMA"]).sum()),
    },
    "pekerjaan": {
        "Belum/Tidak Bekerja": int((job == "BELUM/TIDAK BEKERJA").sum()),
        "Mengurus Rumah Tangga": int((job == "MENGURUS RUMAH TANGGA").sum()),
        "Wiraswasta": int((job == "WIRASWASTA").sum()),
        "Pelajar/Mahasiswa": int((job == "PELAJAR/MAHASISWA").sum()),
        "Buruh Harian Lepas": int((job == "BURUH HARIAN LEPAS").sum()),
        "Karyawan Swasta": int((job == "KARYAWAN SWASTA").sum()),
        "Petani dan Buruh Tani": int(job.isin(["PETANI/PEKEBUN", "BURUH TANI/PERKEBUNAN"]).sum()),
    },
}
public_summary

{'total_penduduk': 15772,
 'laki_laki': 7962,
 'perempuan': 7810,
 'kepala_keluarga': 5379,
 'rt': 33,
 'rw': 15,
 'kelompok_umur': {'0-5 tahun': 409,
  '6-12 tahun': 1523,
  '13-17 tahun': 1127,
  '18-24 tahun': 1775,
  '25-44 tahun': 5258,
  '45-59 tahun': 3391,
  '60 tahun ke atas': 2289},
 'pendidikan': {'Tidak/Belum Sekolah': 5455,
  'Belum Tamat SD': 1263,
  'SD/Sederajat': 4269,
  'SMP/Sederajat': 2565,
  'SMA/Sederajat': 1911,
  'Diploma/Sarjana': 309},
 'pekerjaan': {'Belum/Tidak Bekerja': 4361,
  'Mengurus Rumah Tangga': 4307,
  'Wiraswasta': 2548,
  'Pelajar/Mahasiswa': 1843,
  'Buruh Harian Lepas': 815,
  'Karyawan Swasta': 548,
  'Petani dan Buruh Tani': 932}}

## Takeaways

- Gunakan sheet `ASLI` sebagai sumber agregat publik karena memiliki 15.772 rekaman lengkap dan NIK unik.
- Snapshot data konsisten untuk 11 Oktober 2023; angka ini tidak boleh diberi label sebagai data 2026.
- Gunakan normalisasi yang ada pada notebook sebelum menggabungkan kategori pendidikan dan pekerjaan.
- Jangan memublikasikan kolom nama, NIK, nomor KK, tanggal lahir, atau alamat.